In [23]:
import pandas as pd
import requests

url = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"

data = requests.get(url).json()
df = pd.json_normalize(data)

In [24]:
# Eliminar ID
df = df.drop(columns=["customerID"])

# Convertir TotalCharges
df["account.Charges.Total"] = pd.to_numeric(
    df["account.Charges.Total"],
    errors="coerce"
)

# Eliminar nulos
df = df.dropna()

In [25]:
df["Churn"].unique()

array(['No', 'Yes', ''], dtype=object)

In [26]:
df["Churn"] = df["Churn"].str.strip()
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

In [27]:
df["Churn"].unique()
df["Churn"].isnull().sum()

np.int64(224)

In [28]:
df["Churn"].value_counts(dropna=False)

,count
Churn,
0.0,5163
1.0,1869
NaN,224


In [29]:
df = df.dropna(subset=["Churn"])

In [30]:
df["Churn"].isnull().sum()

np.int64(0)

In [31]:
df.shape

(7032, 20)

#Analisis rapido del desbalance:


In [32]:
df["Churn"].value_counts(normalize=True)

,proportion
Churn,
0.0,0.734215
1.0,0.265785


#Division Train/Test

In [34]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

#Modelo 1

In [39]:
X.dtypes

,0
customer.gender,object
customer.SeniorCitizen,int64
customer.Partner,object
customer.Dependents,object
customer.tenure,int64
phone.PhoneService,object
phone.MultipleLines,object
internet.InternetService,object
internet.OnlineSecurity,object
internet.OnlineBackup,object


In [40]:
df = pd.get_dummies(df, drop_first=True)

In [41]:
df.dtypes.value_counts()

,count
bool,26
float64,3
int64,2


In [42]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [44]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar regresion logistica

In [45]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

#Evaluar regresion logistica

In [46]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_log = log_model.predict(X_test_scaled)

print("Regresión Logística")
print(classification_report(y_test, y_pred_log))

Regresión Logística
              precision    recall  f1-score   support

         0.0       0.84      0.90      0.87      1549
         1.0       0.66      0.54      0.60       561

    accuracy                           0.80      2110
   macro avg       0.75      0.72      0.73      2110
weighted avg       0.79      0.80      0.80      2110



# Random Forest (sin normalizar)

In [47]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

# Evaluar Random Forest

In [48]:
y_pred_rf = rf_model.predict(X_test)

print("Random Forest")
print(classification_report(y_test, y_pred_rf))

Random Forest
              precision    recall  f1-score   support

         0.0       0.83      0.89      0.86      1549
         1.0       0.62      0.48      0.55       561

    accuracy                           0.79      2110
   macro avg       0.73      0.69      0.70      2110
weighted avg       0.77      0.79      0.78      2110



#Importancia de la Regresion Logistica

In [49]:
import pandas as pd

importance_log = pd.DataFrame({
    "Variable": X.columns,
    "Coeficiente": log_model.coef_[0]
})

importance_log = importance_log.sort_values(by="Coeficiente", ascending=False)

importance_log.head(10)

,Variable,Coeficiente
3,account.Charges.Total,0.626025
10,internet.InternetService_Fiber optic,0.594341
21,internet.StreamingTV_Yes,0.201481
28,account.PaymentMethod_Electronic check,0.183835
26,account.PaperlessBilling_Yes,0.180289
23,internet.StreamingMovies_Yes,0.156266
9,phone.MultipleLines_Yes,0.129529
0,customer.SeniorCitizen,0.094955
17,internet.DeviceProtection_Yes,0.038262
8,phone.MultipleLines_No phone service,0.019353


In [50]:
importance_log.tail(10)

,Variable,Coeficiente
18,internet.TechSupport_No internet service,-0.071715
12,internet.OnlineSecurity_No internet service,-0.071715
22,internet.StreamingMovies_No internet service,-0.071715
16,internet.DeviceProtection_No internet service,-0.071715
13,internet.OnlineSecurity_Yes,-0.096265
19,internet.TechSupport_Yes,-0.149677
24,account.Contract_One year,-0.285491
2,account.Charges.Monthly,-0.510416
25,account.Contract_Two year,-0.542835
1,customer.tenure,-1.343554


# Importancia del random Forest

In [51]:
importance_rf = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": rf_model.feature_importances_
})

importance_rf = importance_rf.sort_values(by="Importancia", ascending=False)

importance_rf.head(10)

,Variable,Importancia
3,account.Charges.Total,0.187292
1,customer.tenure,0.173152
2,account.Charges.Monthly,0.172056
28,account.PaymentMethod_Electronic check,0.042456
10,internet.InternetService_Fiber optic,0.037833
4,customer.gender_Male,0.029014
25,account.Contract_Two year,0.028802
13,internet.OnlineSecurity_Yes,0.026217
26,account.PaperlessBilling_Yes,0.025263
19,internet.TechSupport_Yes,0.023652


# **Informe - Predicción de Cancelación (Churn)**

Telecom X

**Objetivo:**

El objetivo de este análisis fie desarrollar modelos predictivos capaces de identificar clientes con alta probabilidad de cancelar servicio (churn), permitiendo a Telecom X anticiparse y aplicar estrategias de retención.

## **Modelos Evaluados:**

Se entrenaron dos modelos de clasificación:

* Regresión Logística (requiere normalización)
* Random Forest (modelo basado en árboles)

Resultados principales:
| Modelo              | Accuracy | Recall (Clase 1) | F1-Score (Clase 1) |
| ------------------- | -------- | ---------------- | ------------------ |
| Regresión Logística | 80%      | 54%              | 0.60               |
| Random Forest       | 79%      | 48%              | 0.55               |

## **Interpretación:**

La Regresión Logística presentó el mejor desempeño general, especialmente en la detención de clientes que cancelan (recall = 54%)

Dado que en problemas de churn es más importante identificar correctamente a los clientes que se van, este modelo fue consolidado el más adecuado.

## **Factores que Más Influyen en la Cancelación:**

Ambos modelos coincidieron en las variables más relevantes:

Factores que AUMENTAN la probabilidad de cancelación:

* Contrato mensual (Month-to-month)

* Servicio de fibra óptica

* Pago mediante electronic check

* Facturación electrónica (paperless billing)

* Cargos totales elevados

Factores que DISMINUYEN la probabilidad de cancelación:

* Mayor tiempo de permanencia (tenure)

* Contratos de uno o dos años

* Tener soporte técnico

* Servicios adicionales de seguridad

El tiempo de permanencia (tenure) fue el factor más determinante. Los clientes con poco tiempo en la empresa presentan mayor riesgo de cancelación.

##**Perfil del cliente con Mayor Riesgo:**

Con base en los resultados, el perfil del cliente con mayor probabilidad de cancelar es:

* Cliente reciente (bajo tenure)

* Contrato mensual

* Servicio de fibra óptica

* Pago con electronic check

* Sin soporte técnico incluido

##**Recomendaciones Estrategicas:**

Con base en los hallazgos, se proponen las siguientes estrategias de retención:

1. Programas de fidelización temprana

Implementar acciones de acompañamiento durante los primeros meses del cliente, periodo donde existe mayor riesgo de cancelación.

2. Incentivar contratos largos

Ofrecer descuentos o beneficios por migrar de contrato mensual a contratos de uno o dos años.

3. Incluir soporte técnico como valor agregado

Promover planes que incluyan soporte técnico, ya que reduce significativamente la probabilidad de churn.

4. Analizar experiencia del servicio de fibra óptica

Investigar posibles problemas en calidad o expectativas no cumplidas en este segmento.

5. Monitoreo preventivo

Crear alertas internas para clientes que cumplan con el perfil de alto riesgo identificado por el modelo.

#**Conclusión Final:**

El análisis demuestra que es posible anticipar la cancelación de clientes con una precisión aceptable (80%). Sin embargo, aún existe margen de mejora en la detección de casos positivos.

El principal hallazgo es que el tiempo de permanencia y el tipo de contrato son determinantes clave en la decisión de cancelación. Implementar estrategias enfocadas en clientes nuevos y en contratos mensuales puede reducir significativamente el churn.